In [3]:
# ======================================================================
# 05_Gauge_Value_Extraction.ipynb
# FULL FIXED VERSION - PROJ DATABASE BYPASS
# ======================================================================
#
# কাজ:
#   1. BMD gauge rainfall CSV পড়বে
#   2. 72টি monthly predictor stack খুঁজবে
#   3. প্রতিটি gauge station location-এ 15টি raster band extract করবে
#   4. PROJ/EPSG database lookup ছাড়াই longitude-latitude ব্যবহার করবে
#   5. extraction failure হলে invalid dataset save করবে না
#   6. CSV, GeoPackage এবং quality-control logs তৈরি করবে
#
# গুরুত্বপূর্ণ:
#   Gauge coordinates এবং aligned raster দুটোই WGS 84 longitude-latitude
#   হওয়ায় coordinates সরাসরি sample করা হচ্ছে।
# ======================================================================


# ======================================================================
# 1. IMPORT LIBRARIES
# ======================================================================

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from shapely.geometry import Point


warnings.filterwarnings("ignore")


# ======================================================================
# 2. PROJECT CONFIGURATION
# ======================================================================

PROJECT_ROOT = Path(
    r"E:\Geospatial\Precipitation Downscaling"
    r"\Precipitation-Downscaling-Khulna"
)


GAUGE_CSV = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "gauge"
    / "bmd_monthly_rainfall_2017_2022.csv"
)


PREDICTOR_STACK_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "predictor_stack"
)


OUTPUT_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "model_data"
)


LOG_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "logs"
)


OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

LOG_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


# ----------------------------------------------------------------------
# Study period
# ----------------------------------------------------------------------

START_YEAR = 2017
END_YEAR = 2022


# ----------------------------------------------------------------------
# Coordinate handling
#
# True রাখা হয়েছে কারণ:
# - Gauge coordinates longitude/latitude
# - Raster grid WGS 84 geographic coordinates
# - Local PROJ database conflict আছে
# ----------------------------------------------------------------------

ASSUME_GAUGE_AND_RASTER_SAME_CRS = True

GAUGE_CRS_TEXT = "EPSG:4326"


# ----------------------------------------------------------------------
# Processing settings
# ----------------------------------------------------------------------

DROP_ROWS_WITH_MISSING_TARGET = True

DROP_ROWS_WITH_ALL_PREDICTORS_MISSING = True

DROP_ROWS_WITH_ANY_MISSING_PREDICTOR = False

CREATE_CLEAN_DATASET = True

OVERWRITE_EXISTING_GPKG = True


# ----------------------------------------------------------------------
# Expected predictor-stack bands
# ----------------------------------------------------------------------

EXPECTED_BAND_NAMES = [
    "CCS",
    "CDR",
    "CHIRPS",
    "ERA5",
    "GSMaP_Gauge",
    "GSMaP_MVK",
    "IMERG",
    "LST",
    "NDVI",
    "PDIR",
    "PERSIANN",
    "Distance_Sea_meter",
    "Land_Aspect_Degree_30m",
    "Land_Khulna_SRTM_DEM",
    "Land_Slope_Degree_30m",
]


# ======================================================================
# 3. HELPER FUNCTIONS
# ======================================================================

def normalize_column_name(column_name):
    """
    Convert column name into lowercase snake_case.
    """

    column_name = str(column_name).strip().lower()

    column_name = re.sub(
        r"[^a-z0-9]+",
        "_",
        column_name
    )

    return column_name.strip("_")


def standardize_gauge_columns(dataframe):
    """
    Standardize gauge-data column names.
    """

    dataframe = dataframe.copy()

    dataframe.columns = [
        normalize_column_name(column)
        for column in dataframe.columns
    ]

    rename_map = {
        "stationid": "station_id",
        "station_id": "station_id",

        "station": "station_name",
        "station_name": "station_name",

        "lat": "latitude",
        "latitude": "latitude",

        "lon": "longitude",
        "long": "longitude",
        "longitude": "longitude",

        "rainfall": "rainfall_mm",
        "rainfall_m": "rainfall_mm",
        "rainfall_mm": "rainfall_mm",

        "yr": "year",
        "year": "year",

        "mon": "month",
        "month": "month",

        "date": "date",
    }

    actual_rename_map = {
        column: rename_map[column]
        for column in dataframe.columns
        if column in rename_map
    }

    dataframe = dataframe.rename(
        columns=actual_rename_map
    )

    return dataframe


def validate_required_columns(dataframe, required_columns):
    """
    Ensure required gauge columns exist.
    """

    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:

        raise ValueError(
            "Gauge CSV-তে required column পাওয়া যায়নি:\n"
            + ", ".join(missing_columns)
        )


def extract_year_month_from_stack_name(filename):
    """
    Extract year and month from stack filename.

    Supported:
        predictor_stack_2017_01.tif
        predictor_stack_2017_1.tif
        2017_01.tif
        201701.tif
    """

    stem = Path(filename).stem

    patterns = [
        r"(?<!\d)(20\d{2})[_\-](0?[1-9]|1[0-2])(?!\d)",
        r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)",
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            stem
        )

        if match:

            year = int(match.group(1))
            month = int(match.group(2))

            if 1 <= month <= 12:
                return year, month

    return None


def build_stack_index(stack_folder):
    """
    Build predictor stack index.

    Returns:
        stack_index[(year, month)] = stack_path
    """

    stack_folder = Path(stack_folder)

    stack_files = sorted([
        path
        for path in stack_folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in [".tif", ".tiff"]
    ])

    stack_index = {}
    skipped_files = []
    duplicate_files = []

    for stack_path in stack_files:

        year_month = extract_year_month_from_stack_name(
            stack_path.name
        )

        if year_month is None:

            skipped_files.append(
                stack_path
            )

            continue

        if year_month in stack_index:

            duplicate_files.append({
                "Year": year_month[0],
                "Month": year_month[1],
                "First_File": str(
                    stack_index[year_month]
                ),
                "Duplicate_File": str(
                    stack_path
                ),
            })

            continue

        stack_index[year_month] = stack_path

    return (
        stack_index,
        skipped_files,
        duplicate_files
    )


def sanitize_band_name(name, band_number):
    """
    Create safe band column names.
    """

    if name is None:
        name = f"band_{band_number}"

    name = str(name).strip()

    name = re.sub(
        r"\s+",
        "_",
        name
    )

    name = re.sub(
        r"[^A-Za-z0-9_]+",
        "",
        name
    )

    if not name:
        name = f"band_{band_number}"

    return name


def make_unique_names(names):
    """
    Ensure all output band names are unique.
    """

    output_names = []
    name_counter = {}

    for name in names:

        if name not in name_counter:

            name_counter[name] = 1
            output_names.append(name)

        else:

            name_counter[name] += 1

            output_names.append(
                f"{name}_{name_counter[name]}"
            )

    return output_names


def get_band_names(raster_dataset):
    """
    Read raster band descriptions.

    Uses EXPECTED_BAND_NAMES if descriptions are absent.
    """

    descriptions = list(
        raster_dataset.descriptions
    )

    band_names = []

    for band_number in range(
        1,
        raster_dataset.count + 1
    ):

        description = descriptions[
            band_number - 1
        ]

        if description is None:

            if band_number <= len(EXPECTED_BAND_NAMES):

                description = EXPECTED_BAND_NAMES[
                    band_number - 1
                ]

            else:

                description = (
                    f"band_{band_number}"
                )

        clean_name = sanitize_band_name(
            description,
            band_number
        )

        band_names.append(
            clean_name
        )

    return make_unique_names(
        band_names
    )


def point_inside_bounds(x, y, bounds):
    """
    Check whether a coordinate lies inside raster bounds.
    """

    return (
        bounds.left <= x <= bounds.right
        and
        bounds.bottom <= y <= bounds.top
    )


def prepare_raster_coordinates(
    longitude,
    latitude
):
    """
    Return longitude and latitude directly.

    This function intentionally avoids:
        CRS.from_epsg()
        CRS.from_user_input()
        rasterio.warp.transform()

    because the current Conda environment has a PROJ database conflict.
    """

    longitude = float(longitude)
    latitude = float(latitude)

    if not np.isfinite(longitude):

        raise ValueError(
            f"Invalid longitude: {longitude}"
        )

    if not np.isfinite(latitude):

        raise ValueError(
            f"Invalid latitude: {latitude}"
        )

    if not ASSUME_GAUGE_AND_RASTER_SAME_CRS:

        raise RuntimeError(
            "Coordinate transformation is disabled because "
            "the local PROJ database is incompatible."
        )

    return longitude, latitude


def clean_sample_value(value, nodata_value):
    """
    Convert nodata, NaN and infinity to np.nan.
    """

    try:

        value = float(value)

    except Exception:

        return np.nan

    if not np.isfinite(value):
        return np.nan

    if nodata_value is not None:

        try:

            if np.isclose(
                value,
                float(nodata_value),
                equal_nan=False
            ):

                return np.nan

        except Exception:
            pass

    return value


def extract_stack_values(
    stack_path,
    station_dataframe
):
    """
    Extract every predictor-stack band at all station locations.
    """

    extraction_rows = []
    extraction_logs = []

    with rasterio.open(stack_path) as src:

        band_names = get_band_names(
            src
        )

        raster_bounds = src.bounds
        raster_nodata = src.nodata

        raster_crs_text = (
            str(src.crs)
            if src.crs is not None
            else "Unknown"
        )

        for _, station_row in station_dataframe.iterrows():

            station_id = station_row["station_id"]

            longitude = station_row["longitude"]
            latitude = station_row["latitude"]

            output_row = station_row.to_dict()

            try:

                raster_x, raster_y = (
                    prepare_raster_coordinates(
                        longitude=longitude,
                        latitude=latitude
                    )
                )

                output_row["raster_x"] = raster_x
                output_row["raster_y"] = raster_y

                inside_bounds = point_inside_bounds(
                    raster_x,
                    raster_y,
                    raster_bounds
                )

                output_row[
                    "inside_raster_bounds"
                ] = inside_bounds

                if not inside_bounds:

                    for band_name in band_names:
                        output_row[band_name] = np.nan

                    extraction_rows.append(
                        output_row
                    )

                    extraction_logs.append({
                        "Station_ID": station_id,
                        "Stack_File": str(stack_path),
                        "Status": "Outside raster bounds",
                        "Longitude": longitude,
                        "Latitude": latitude,
                        "Raster_X": raster_x,
                        "Raster_Y": raster_y,
                        "Raster_CRS": raster_crs_text,
                        "Raster_Bounds": str(raster_bounds),
                        "Valid_Predictor_Count": 0,
                        "Total_Bands": len(band_names),
                        "Error": (
                            "Gauge coordinate lies outside "
                            "the raster extent."
                        ),
                    })

                    continue

                sampled_values = next(
                    src.sample(
                        [
                            (
                                raster_x,
                                raster_y
                            )
                        ]
                    )
                )

                if len(sampled_values) != len(
                    band_names
                ):

                    raise RuntimeError(
                        f"Sampled band count mismatch: "
                        f"{len(sampled_values)} != "
                        f"{len(band_names)}"
                    )

                for band_index, band_name in enumerate(
                    band_names
                ):

                    output_row[band_name] = (
                        clean_sample_value(
                            sampled_values[band_index],
                            raster_nodata
                        )
                    )

                valid_predictor_count = sum(
                    pd.notna(
                        output_row[band_name]
                    )
                    for band_name in band_names
                )

                extraction_rows.append(
                    output_row
                )

                extraction_logs.append({
                    "Station_ID": station_id,
                    "Stack_File": str(stack_path),
                    "Status": "Success",
                    "Longitude": longitude,
                    "Latitude": latitude,
                    "Raster_X": raster_x,
                    "Raster_Y": raster_y,
                    "Raster_CRS": raster_crs_text,
                    "Raster_Bounds": str(raster_bounds),
                    "Valid_Predictor_Count": (
                        valid_predictor_count
                    ),
                    "Total_Bands": len(band_names),
                    "Error": "",
                })

            except Exception as error:

                output_row["raster_x"] = np.nan
                output_row["raster_y"] = np.nan

                output_row[
                    "inside_raster_bounds"
                ] = False

                for band_name in band_names:
                    output_row[band_name] = np.nan

                extraction_rows.append(
                    output_row
                )

                extraction_logs.append({
                    "Station_ID": station_id,
                    "Stack_File": str(stack_path),
                    "Status": "Failed",
                    "Longitude": longitude,
                    "Latitude": latitude,
                    "Raster_X": np.nan,
                    "Raster_Y": np.nan,
                    "Raster_CRS": raster_crs_text,
                    "Raster_Bounds": str(raster_bounds),
                    "Valid_Predictor_Count": 0,
                    "Total_Bands": len(band_names),
                    "Error": str(error),
                })

    return (
        extraction_rows,
        extraction_logs,
        band_names
    )


def assign_season(month):
    """
    Bangladesh seasonal grouping.
    """

    if month in [12, 1, 2]:
        return "Winter"

    if month in [3, 4, 5]:
        return "Pre_Monsoon"

    if month in [6, 7, 8, 9]:
        return "Monsoon"

    return "Post_Monsoon"


# ======================================================================
# 4. INITIAL VALIDATION
# ======================================================================

print("=" * 78)
print("GAUGE VALUE EXTRACTION FROM PREDICTOR STACKS")
print("=" * 78)

print(f"Project root           : {PROJECT_ROOT}")
print(f"Gauge CSV              : {GAUGE_CSV}")
print(f"Predictor stack folder : {PREDICTOR_STACK_FOLDER}")
print(f"Output folder          : {OUTPUT_FOLDER}")
print(f"Log folder             : {LOG_FOLDER}")
print()


if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        f"Project root পাওয়া যায়নি:\n"
        f"{PROJECT_ROOT}"
    )


if not GAUGE_CSV.exists():

    raise FileNotFoundError(
        f"Gauge CSV পাওয়া যায়নি:\n"
        f"{GAUGE_CSV}"
    )


if not PREDICTOR_STACK_FOLDER.exists():

    raise FileNotFoundError(
        f"Predictor stack folder পাওয়া যায়নি:\n"
        f"{PREDICTOR_STACK_FOLDER}"
    )


# ======================================================================
# 5. LOAD AND PREPARE GAUGE DATA
# ======================================================================

print("=" * 78)
print("LOADING GAUGE DATA")
print("=" * 78)


gauge_df = pd.read_csv(
    GAUGE_CSV
)


print(
    f"Original rows    : "
    f"{len(gauge_df)}"
)

print(
    f"Original columns : "
    f"{list(gauge_df.columns)}"
)


gauge_df = standardize_gauge_columns(
    gauge_df
)


required_columns = [
    "station_id",
    "station_name",
    "latitude",
    "longitude",
    "year",
    "month",
    "rainfall_mm",
]


validate_required_columns(
    gauge_df,
    required_columns
)


numeric_columns = [
    "latitude",
    "longitude",
    "year",
    "month",
    "rainfall_mm",
]


for column in numeric_columns:

    gauge_df[column] = pd.to_numeric(
        gauge_df[column],
        errors="coerce"
    )


gauge_df["station_id"] = (
    gauge_df["station_id"]
    .astype(str)
    .str.strip()
)


gauge_df["station_name"] = (
    gauge_df["station_name"]
    .astype(str)
    .str.strip()
)


original_row_count = len(
    gauge_df
)


gauge_df = gauge_df.dropna(
    subset=[
        "station_id",
        "latitude",
        "longitude",
        "year",
        "month",
    ]
).copy()


gauge_df["year"] = (
    gauge_df["year"]
    .astype(int)
)


gauge_df["month"] = (
    gauge_df["month"]
    .astype(int)
)


gauge_df = gauge_df[
    gauge_df["year"].between(
        START_YEAR,
        END_YEAR
    )
].copy()


gauge_df = gauge_df[
    gauge_df["month"].between(
        1,
        12
    )
].copy()


# ----------------------------------------------------------------------
# Basic coordinate validation
# ----------------------------------------------------------------------

invalid_longitude_count = int(
    (
        ~gauge_df["longitude"].between(
            -180,
            180
        )
    ).sum()
)


invalid_latitude_count = int(
    (
        ~gauge_df["latitude"].between(
            -90,
            90
        )
    ).sum()
)


if invalid_longitude_count > 0:

    raise ValueError(
        f"{invalid_longitude_count}টি invalid longitude পাওয়া গেছে।"
    )


if invalid_latitude_count > 0:

    raise ValueError(
        f"{invalid_latitude_count}টি invalid latitude পাওয়া গেছে।"
    )


gauge_df["date"] = pd.to_datetime(
    dict(
        year=gauge_df["year"],
        month=gauge_df["month"],
        day=1
    ),
    errors="coerce"
)


gauge_df["month_id"] = (
    gauge_df["year"].astype(str)
    + "_"
    + gauge_df["month"]
    .astype(str)
    .str.zfill(2)
)


gauge_df = gauge_df.sort_values(
    [
        "year",
        "month",
        "station_id"
    ]
).reset_index(
    drop=True
)


duplicate_count = int(
    gauge_df.duplicated(
        subset=[
            "station_id",
            "year",
            "month"
        ]
    ).sum()
)


print()
print(
    f"Prepared rows           : "
    f"{len(gauge_df)}"
)

print(
    f"Removed invalid rows    : "
    f"{original_row_count - len(gauge_df)}"
)

print(
    f"Unique stations         : "
    f"{gauge_df['station_id'].nunique()}"
)

print(
    f"Duplicate station-month : "
    f"{duplicate_count}"
)

print(
    f"Missing rainfall        : "
    f"{gauge_df['rainfall_mm'].isna().sum()}"
)

print(
    f"Longitude range         : "
    f"{gauge_df['longitude'].min()} to "
    f"{gauge_df['longitude'].max()}"
)

print(
    f"Latitude range          : "
    f"{gauge_df['latitude'].min()} to "
    f"{gauge_df['latitude'].max()}"
)

print(
    f"Period                  : "
    f"{gauge_df['date'].min()} to "
    f"{gauge_df['date'].max()}"
)


if duplicate_count > 0:

    raise ValueError(
        "Gauge dataset-এ duplicate station-year-month "
        "record পাওয়া গেছে।"
    )


# ======================================================================
# 6. INDEX PREDICTOR STACKS
# ======================================================================

print()
print("=" * 78)
print("INDEXING PREDICTOR STACKS")
print("=" * 78)


(
    stack_index,
    skipped_stack_files,
    duplicate_stack_files
) = build_stack_index(
    PREDICTOR_STACK_FOLDER
)


print(
    f"Indexed stacks   : "
    f"{len(stack_index)}"
)

print(
    f"Skipped files    : "
    f"{len(skipped_stack_files)}"
)

print(
    f"Duplicate months : "
    f"{len(duplicate_stack_files)}"
)


expected_months = [
    (year, month)
    for year in range(
        START_YEAR,
        END_YEAR + 1
    )
    for month in range(
        1,
        13
    )
]


missing_stack_months = [
    (year, month)
    for year, month in expected_months
    if (year, month) not in stack_index
]


if missing_stack_months:

    print()
    print("Missing predictor stacks:")

    for year, month in missing_stack_months:

        print(
            f"  {year}_{month:02d}"
        )

    raise FileNotFoundError(
        f"{len(missing_stack_months)}টি "
        f"predictor stack missing আছে।"
    )


print(
    f"All {len(expected_months)} expected stacks found."
)


# ======================================================================
# 7. TEST FIRST RASTER AND GAUGE COORDINATES
# ======================================================================

print()
print("=" * 78)
print("PRE-EXTRACTION COORDINATE CHECK")
print("=" * 78)


first_stack_path = stack_index[
    expected_months[0]
]


with rasterio.open(first_stack_path) as test_src:

    print(
        f"Test raster : "
        f"{first_stack_path.name}"
    )

    print(
        f"Raster CRS  : "
        f"{test_src.crs}"
    )

    print(
        f"Raster bounds: "
        f"{test_src.bounds}"
    )

    print(
        f"Raster bands : "
        f"{test_src.count}"
    )

    unique_station_coordinates = (
        gauge_df[
            [
                "station_id",
                "station_name",
                "longitude",
                "latitude"
            ]
        ]
        .drop_duplicates(
            subset=[
                "station_id"
            ]
        )
        .sort_values(
            "station_id"
        )
    )

    print()
    print("Gauge station coordinate check:")

    for _, station in unique_station_coordinates.iterrows():

        inside = point_inside_bounds(
            station["longitude"],
            station["latitude"],
            test_src.bounds
        )

        print(
            f"{station['station_id']} | "
            f"{station['station_name']} | "
            f"Lon={station['longitude']} | "
            f"Lat={station['latitude']} | "
            f"Inside={inside}"
        )


# ======================================================================
# 8. EXTRACT GAUGE VALUES
# ======================================================================

print()
print("=" * 78)
print("EXTRACTING GAUGE-LOCATION PREDICTOR VALUES")
print("=" * 78)


all_extracted_rows = []
all_extraction_logs = []
monthly_processing_records = []

final_band_names = None

successful_months = 0
failed_months = 0


for year, month in expected_months:

    month_id = f"{year}_{month:02d}"

    print()
    print("-" * 78)
    print(f"Processing: {month_id}")
    print("-" * 78)

    stack_path = stack_index[
        (year, month)
    ]

    month_gauge_df = gauge_df[
        (gauge_df["year"] == year)
        &
        (gauge_df["month"] == month)
    ].copy()


    expected_station_count = len(
        month_gauge_df
    )


    if month_gauge_df.empty:

        failed_months += 1

        print(
            f"FAILED: No gauge records for {month_id}"
        )

        monthly_processing_records.append({
            "Year": year,
            "Month": month,
            "Month_ID": month_id,
            "Stack_File": str(stack_path),
            "Expected_Stations": 0,
            "Successful_Stations": 0,
            "Outside_Stations": 0,
            "Failed_Stations": 0,
            "Status": "Failed",
            "Error": "No gauge records",
        })

        continue


    extraction_rows = []
    extraction_logs = []
    band_names = []

    success_count = 0
    outside_count = 0
    failure_count = 0


    try:

        (
            extraction_rows,
            extraction_logs,
            band_names
        ) = extract_stack_values(
            stack_path=stack_path,
            station_dataframe=month_gauge_df
        )


        if final_band_names is None:

            final_band_names = band_names


        if band_names != final_band_names:

            raise ValueError(
                f"Band names/order mismatch in "
                f"{stack_path.name}"
            )


        for log_record in extraction_logs:

            log_record["Year"] = year
            log_record["Month"] = month
            log_record["Month_ID"] = month_id


        success_count = sum(
            record["Status"] == "Success"
            for record in extraction_logs
        )


        outside_count = sum(
            record["Status"] == "Outside raster bounds"
            for record in extraction_logs
        )


        failure_count = sum(
            record["Status"] == "Failed"
            for record in extraction_logs
        )


        print(f"Stack     : {stack_path.name}")
        print(f"Stations  : {expected_station_count}")
        print(f"Success   : {success_count}")
        print(f"Outside   : {outside_count}")
        print(f"Failed    : {failure_count}")
        print(f"Bands     : {len(band_names)}")


        problem_logs = [
            record
            for record in extraction_logs
            if record["Status"] != "Success"
        ]


        if problem_logs:

            print()
            print("Station extraction problems:")

            for record in problem_logs:

                print(
                    f"  Station {record['Station_ID']} | "
                    f"{record['Status']} | "
                    f"{record['Error']}"
                )


        if success_count != expected_station_count:

            raise RuntimeError(
                f"{month_id}: expected "
                f"{expected_station_count} successful "
                f"station extractions, but got "
                f"{success_count}. "
                f"Outside={outside_count}, "
                f"Failed={failure_count}"
            )


        # Month completely successful হলে data append হবে
        all_extracted_rows.extend(
            extraction_rows
        )

        all_extraction_logs.extend(
            extraction_logs
        )


        successful_months += 1


        monthly_processing_records.append({
            "Year": year,
            "Month": month,
            "Month_ID": month_id,
            "Stack_File": str(stack_path),
            "Expected_Stations": expected_station_count,
            "Successful_Stations": success_count,
            "Outside_Stations": outside_count,
            "Failed_Stations": failure_count,
            "Status": "Success",
            "Error": "",
        })


    except Exception as error:

        failed_months += 1

        all_extraction_logs.extend(
            extraction_logs
        )

        print()
        print(f"FAILED MONTH: {month_id}")
        print(f"ERROR       : {error}")


        monthly_processing_records.append({
            "Year": year,
            "Month": month,
            "Month_ID": month_id,
            "Stack_File": str(stack_path),
            "Expected_Stations": expected_station_count,
            "Successful_Stations": success_count,
            "Outside_Stations": outside_count,
            "Failed_Stations": failure_count,
            "Status": "Failed",
            "Error": str(error),
        })


# ======================================================================
# 9. MONTHLY EXTRACTION SUMMARY
# ======================================================================

print()
print("=" * 78)
print("MONTHLY EXTRACTION SUMMARY")
print("=" * 78)

print(
    f"Expected months   : "
    f"{len(expected_months)}"
)

print(
    f"Successful months : "
    f"{successful_months}"
)

print(
    f"Failed months     : "
    f"{failed_months}"
)


monthly_processing_df = pd.DataFrame(
    monthly_processing_records
)


MONTHLY_PROCESSING_LOG = (
    LOG_FOLDER
    / "gauge_value_monthly_processing.csv"
)


monthly_processing_df.to_csv(
    MONTHLY_PROCESSING_LOG,
    index=False
)


extraction_log_df_early = pd.DataFrame(
    all_extraction_logs
)


EARLY_EXTRACTION_LOG = (
    LOG_FOLDER
    / "gauge_value_extraction_log.csv"
)


extraction_log_df_early.to_csv(
    EARLY_EXTRACTION_LOG,
    index=False
)


if failed_months > 0:

    failed_month_df = monthly_processing_df[
        monthly_processing_df["Status"] == "Failed"
    ].copy()

    print()
    print("Failed months:")

    print(
        failed_month_df[
            [
                "Month_ID",
                "Successful_Stations",
                "Outside_Stations",
                "Failed_Stations",
                "Error"
            ]
        ].to_string(
            index=False
        )
    )

    raise RuntimeError(
        f"{failed_months}টি month extraction failed হয়েছে। "
        f"Invalid model dataset save করা হয়নি। "
        f"Log দেখুন:\n{MONTHLY_PROCESSING_LOG}\n"
        f"{EARLY_EXTRACTION_LOG}"
    )


# ======================================================================
# 10. CREATE FINAL MODEL DATAFRAME
# ======================================================================

print()
print("=" * 78)
print("CREATING FINAL MODEL DATASET")
print("=" * 78)


if not all_extracted_rows:

    raise RuntimeError(
        "কোনো predictor value successfully extract হয়নি।"
    )


model_df = pd.DataFrame(
    all_extracted_rows
)


model_df["date"] = pd.to_datetime(
    model_df["date"],
    errors="coerce"
)


# ----------------------------------------------------------------------
# Temporal features
# ----------------------------------------------------------------------

model_df["month_sin"] = np.sin(
    2
    * np.pi
    * model_df["month"]
    / 12
)


model_df["month_cos"] = np.cos(
    2
    * np.pi
    * model_df["month"]
    / 12
)


model_df["season"] = model_df[
    "month"
].apply(
    assign_season
)


# ----------------------------------------------------------------------
# Predictor completeness
# ----------------------------------------------------------------------

predictor_columns = [
    column
    for column in final_band_names
    if column in model_df.columns
]


if len(predictor_columns) != len(
    EXPECTED_BAND_NAMES
):

    print()
    print(
        "WARNING: Predictor column count differs "
        "from the expected band count."
    )

    print(
        f"Expected: {len(EXPECTED_BAND_NAMES)}"
    )

    print(
        f"Found   : {len(predictor_columns)}"
    )


model_df["valid_predictor_count"] = (
    model_df[
        predictor_columns
    ]
    .notna()
    .sum(axis=1)
)


model_df["missing_predictor_count"] = (
    model_df[
        predictor_columns
    ]
    .isna()
    .sum(axis=1)
)


model_df["all_predictors_available"] = (
    model_df["missing_predictor_count"]
    == 0
)


# ----------------------------------------------------------------------
# Reorder columns
# ----------------------------------------------------------------------

metadata_columns = [
    "station_id",
    "station_name",
    "latitude",
    "longitude",
    "year",
    "month",
    "date",
    "month_id",
    "season",
    "month_sin",
    "month_cos",
    "rainfall_mm",
]


quality_columns = [
    "raster_x",
    "raster_y",
    "inside_raster_bounds",
    "valid_predictor_count",
    "missing_predictor_count",
    "all_predictors_available",
]


ordered_columns = (
    metadata_columns
    + predictor_columns
    + quality_columns
)


remaining_columns = [
    column
    for column in model_df.columns
    if column not in ordered_columns
]


model_df = model_df[
    ordered_columns
    + remaining_columns
]


model_df = model_df.sort_values(
    [
        "year",
        "month",
        "station_id"
    ]
).reset_index(
    drop=True
)


print(
    f"Final rows          : "
    f"{len(model_df)}"
)

print(
    f"Predictor columns   : "
    f"{len(predictor_columns)}"
)

print(
    f"Complete rows       : "
    f"{model_df['all_predictors_available'].sum()}"
)

print(
    f"Rows with missing   : "
    f"{(~model_df['all_predictors_available']).sum()}"
)

print(
    f"Outside bounds rows : "
    f"{(~model_df['inside_raster_bounds']).sum()}"
)


# ======================================================================
# 11. FINAL QUALITY VALIDATION
# ======================================================================

expected_row_count = len(
    gauge_df
)


if len(model_df) != expected_row_count:

    raise RuntimeError(
        f"Extracted row count mismatch: "
        f"{len(model_df)} != {expected_row_count}"
    )


duplicate_output_rows = int(
    model_df.duplicated(
        subset=[
            "station_id",
            "year",
            "month"
        ]
    ).sum()
)


if duplicate_output_rows > 0:

    raise RuntimeError(
        f"{duplicate_output_rows}টি duplicate "
        f"station-month row পাওয়া গেছে।"
    )


outside_row_count = int(
    (
        ~model_df[
            "inside_raster_bounds"
        ]
    ).sum()
)


if outside_row_count > 0:

    raise RuntimeError(
        f"{outside_row_count}টি row raster "
        f"bounds-এর বাইরে।"
    )


all_missing_predictor_rows = int(
    (
        model_df[
            "valid_predictor_count"
        ]
        == 0
    ).sum()
)


if all_missing_predictor_rows > 0:

    raise RuntimeError(
        f"{all_missing_predictor_rows}টি row-তে "
        f"সব predictor missing।"
    )


# ======================================================================
# 12. CREATE CLEAN ML DATASET
# ======================================================================

clean_model_df = model_df.copy()


if DROP_ROWS_WITH_MISSING_TARGET:

    clean_model_df = clean_model_df.dropna(
        subset=[
            "rainfall_mm"
        ]
    ).copy()


if DROP_ROWS_WITH_ALL_PREDICTORS_MISSING:

    clean_model_df = clean_model_df[
        clean_model_df[
            "valid_predictor_count"
        ]
        > 0
    ].copy()


if DROP_ROWS_WITH_ANY_MISSING_PREDICTOR:

    clean_model_df = clean_model_df.dropna(
        subset=predictor_columns
    ).copy()


clean_model_df = clean_model_df.reset_index(
    drop=True
)


print()
print(
    f"Clean ML rows       : "
    f"{len(clean_model_df)}"
)


# ======================================================================
# 13. CREATE GEODATAFRAME
# ======================================================================

geometry = [
    Point(
        longitude,
        latitude
    )
    for longitude, latitude in zip(
        model_df["longitude"],
        model_df["latitude"]
    )
]


# CRS string ব্যবহার করছি।
# এখানে CRS.from_epsg() call করা হচ্ছে না।
model_gdf = gpd.GeoDataFrame(
    model_df.copy(),
    geometry=geometry
)


# CRS assignment-এও PROJ error হতে পারে।
# তাই try/except রাখা হয়েছে।
try:

    model_gdf = model_gdf.set_crs(
        GAUGE_CRS_TEXT,
        allow_override=True
    )

    gpkg_crs_assigned = True

except Exception as error:

    print()
    print(
        "WARNING: GeoDataFrame CRS assign করা যায়নি "
        "কারণ local PROJ database incompatible."
    )

    print(
        f"CRS error: {error}"
    )

    print(
        "CSV files তৈরি হবে, কিন্তু GeoPackage save "
        "skip করা হবে।"
    )

    gpkg_crs_assigned = False


# ======================================================================
# 14. CREATE SUMMARY TABLES
# ======================================================================

extraction_log_df = pd.DataFrame(
    all_extraction_logs
)


missing_summary_records = []


for predictor in predictor_columns:

    valid_count = int(
        model_df[
            predictor
        ].notna().sum()
    )

    missing_count = int(
        model_df[
            predictor
        ].isna().sum()
    )

    missing_percentage = (
        missing_count
        / len(model_df)
        * 100
    )

    missing_summary_records.append({
        "Predictor": predictor,
        "Valid_Values": valid_count,
        "Missing_Values": missing_count,
        "Missing_Percentage": round(
            missing_percentage,
            4
        ),
    })


missing_summary_df = pd.DataFrame(
    missing_summary_records
)


station_summary_df = (
    model_df
    .groupby(
        [
            "station_id",
            "station_name"
        ],
        as_index=False
    )
    .agg(
        records=(
            "rainfall_mm",
            "size"
        ),
        valid_rainfall=(
            "rainfall_mm",
            "count"
        ),
        mean_rainfall_mm=(
            "rainfall_mm",
            "mean"
        ),
        min_rainfall_mm=(
            "rainfall_mm",
            "min"
        ),
        max_rainfall_mm=(
            "rainfall_mm",
            "max"
        ),
        complete_predictor_rows=(
            "all_predictors_available",
            "sum"
        ),
        mean_valid_predictors=(
            "valid_predictor_count",
            "mean"
        ),
    )
)


station_summary_df[
    "predictor_completeness_percent"
] = (
    station_summary_df[
        "complete_predictor_rows"
    ]
    /
    station_summary_df[
        "records"
    ]
    * 100
)


summary_df = pd.DataFrame([
    {
        "Metric": "Expected months",
        "Value": len(expected_months),
    },
    {
        "Metric": "Successful months",
        "Value": successful_months,
    },
    {
        "Metric": "Failed months",
        "Value": failed_months,
    },
    {
        "Metric": "Gauge input rows",
        "Value": len(gauge_df),
    },
    {
        "Metric": "Extracted rows",
        "Value": len(model_df),
    },
    {
        "Metric": "Clean model rows",
        "Value": len(clean_model_df),
    },
    {
        "Metric": "Unique stations",
        "Value": model_df[
            "station_id"
        ].nunique(),
    },
    {
        "Metric": "Predictor bands",
        "Value": len(
            predictor_columns
        ),
    },
    {
        "Metric": "Complete predictor rows",
        "Value": int(
            model_df[
                "all_predictors_available"
            ].sum()
        ),
    },
    {
        "Metric": "Rows with missing predictor",
        "Value": int(
            (
                ~model_df[
                    "all_predictors_available"
                ]
            ).sum()
        ),
    },
    {
        "Metric": "Rows outside raster bounds",
        "Value": int(
            (
                ~model_df[
                    "inside_raster_bounds"
                ]
            ).sum()
        ),
    },
    {
        "Metric": "GeoPackage CRS assigned",
        "Value": gpkg_crs_assigned,
    },
])


# ======================================================================
# 15. OUTPUT PATHS
# ======================================================================

FULL_CSV_OUTPUT = (
    OUTPUT_FOLDER
    / "gauge_predictor_dataset_2017_2022.csv"
)


CLEAN_CSV_OUTPUT = (
    OUTPUT_FOLDER
    / "gauge_predictor_dataset_clean.csv"
)


GPKG_OUTPUT = (
    OUTPUT_FOLDER
    / "gauge_predictor_dataset_2017_2022.gpkg"
)


EXTRACTION_LOG_OUTPUT = (
    LOG_FOLDER
    / "gauge_value_extraction_log.csv"
)


SUMMARY_LOG_OUTPUT = (
    LOG_FOLDER
    / "gauge_value_extraction_summary.csv"
)


MISSING_SUMMARY_OUTPUT = (
    LOG_FOLDER
    / "gauge_value_missing_summary.csv"
)


STATION_SUMMARY_OUTPUT = (
    LOG_FOLDER
    / "gauge_value_station_summary.csv"
)


DUPLICATE_STACK_OUTPUT = (
    LOG_FOLDER
    / "gauge_value_duplicate_stacks.csv"
)


SKIPPED_STACK_OUTPUT = (
    LOG_FOLDER
    / "gauge_value_unrecognized_stack_files.csv"
)


# ======================================================================
# 16. SAVE OUTPUT FILES
# ======================================================================

model_df.to_csv(
    FULL_CSV_OUTPUT,
    index=False
)


if CREATE_CLEAN_DATASET:

    clean_model_df.to_csv(
        CLEAN_CSV_OUTPUT,
        index=False
    )


gpkg_saved = False


if gpkg_crs_assigned:

    try:

        if (
            GPKG_OUTPUT.exists()
            and OVERWRITE_EXISTING_GPKG
        ):

            GPKG_OUTPUT.unlink()


        model_gdf.to_file(
            GPKG_OUTPUT,
            layer="gauge_predictor_data",
            driver="GPKG"
        )

        gpkg_saved = True

    except Exception as error:

        print()
        print(
            "WARNING: GeoPackage save করা যায়নি।"
        )

        print(
            f"GeoPackage error: {error}"
        )

        print(
            "CSV outputs ঠিকভাবে save হয়েছে।"
        )


extraction_log_df.to_csv(
    EXTRACTION_LOG_OUTPUT,
    index=False
)


summary_df.to_csv(
    SUMMARY_LOG_OUTPUT,
    index=False
)


missing_summary_df.to_csv(
    MISSING_SUMMARY_OUTPUT,
    index=False
)


station_summary_df.to_csv(
    STATION_SUMMARY_OUTPUT,
    index=False
)


pd.DataFrame(
    duplicate_stack_files
).to_csv(
    DUPLICATE_STACK_OUTPUT,
    index=False
)


pd.DataFrame({
    "Skipped_File": [
        str(path)
        for path in skipped_stack_files
    ]
}).to_csv(
    SKIPPED_STACK_OUTPUT,
    index=False
)


# ======================================================================
# 17. FINAL QUALITY CHECK
# ======================================================================

print()
print("=" * 78)
print("FINAL QUALITY CHECK")
print("=" * 78)

print(
    f"Duplicate station-month rows : "
    f"{duplicate_output_rows}"
)

print(
    f"Missing target rainfall      : "
    f"{model_df['rainfall_mm'].isna().sum()}"
)

print(
    f"All predictors complete      : "
    f"{model_df['all_predictors_available'].sum()}"
)

print(
    f"Rows with missing predictors : "
    f"{(~model_df['all_predictors_available']).sum()}"
)

print(
    f"Rows outside raster extent   : "
    f"{(~model_df['inside_raster_bounds']).sum()}"
)


# ======================================================================
# 18. DISPLAY RESULTS
# ======================================================================

print()
print("=" * 78)
print("FINAL DATASET SAMPLE")
print("=" * 78)

display(
    model_df.head(12)
)


print()
print("=" * 78)
print("MISSING VALUE SUMMARY")
print("=" * 78)

display(
    missing_summary_df
)


print()
print("=" * 78)
print("STATION SUMMARY")
print("=" * 78)

display(
    station_summary_df
)


# ======================================================================
# 19. COMPLETION MESSAGE
# ======================================================================

print()
print("=" * 78)
print("GAUGE VALUE EXTRACTION COMPLETED")
print("=" * 78)

print(
    f"Successful months : "
    f"{successful_months}"
)

print(
    f"Failed months     : "
    f"{failed_months}"
)

print(
    f"Final rows        : "
    f"{len(model_df)}"
)

print(
    f"Clean ML rows     : "
    f"{len(clean_model_df)}"
)

print(
    f"Predictor columns : "
    f"{len(predictor_columns)}"
)

print(
    f"GeoPackage saved  : "
    f"{gpkg_saved}"
)


print()
print("Output files:")

print(f"1. {FULL_CSV_OUTPUT}")
print(f"2. {CLEAN_CSV_OUTPUT}")

if gpkg_saved:
    print(f"3. {GPKG_OUTPUT}")
else:
    print(
        "3. GeoPackage skipped due to local PROJ conflict"
    )


print()
print("Log files:")

print(f"1. {EXTRACTION_LOG_OUTPUT}")
print(f"2. {SUMMARY_LOG_OUTPUT}")
print(f"3. {MISSING_SUMMARY_OUTPUT}")
print(f"4. {STATION_SUMMARY_OUTPUT}")
print(f"5. {MONTHLY_PROCESSING_LOG}")


print()
print("Predictor columns:")

for number, predictor in enumerate(
    predictor_columns,
    start=1
):

    print(
        f"{number:02d}. {predictor}"
    )

GAUGE VALUE EXTRACTION FROM PREDICTOR STACKS
Project root           : E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna
Gauge CSV              : E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\gauge\bmd_monthly_rainfall_2017_2022.csv
Predictor stack folder : E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\predictor_stack
Output folder          : E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\model_data
Log folder             : E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\logs

LOADING GAUGE DATA
Original rows    : 432
Original columns : ['station_id', 'station_name', 'latitude', 'longitude', 'year', 'month', 'date', 'rainfall_mm']

Prepared rows           : 432
Removed invalid rows    : 0
Unique stations         : 6
Duplicate station-month : 0
Missing rainfall        : 0
Longitude range         : 89.30

,station_id,station_name,latitude,longitude,year,month,date,month_id,season,month_sin,...,Distance_Sea_meter,Land_Aspect_Degree_30m,Land_Khulna_SRTM_DEM,Land_Slope_Degree_30m,raster_x,raster_y,inside_raster_bounds,valid_predictor_count,missing_predictor_count,all_predictors_available
0,CL503,Chalna,22.6012,89.5195,2017,1,2017-01-01,2017_01,Winter,0.500000,...,31467.640625,108.518059,3.491428,1.720395,89.5195,22.6012,True,13,2,False
1,CL504,Dumuria,22.8093,89.4145,2017,1,2017-01-01,2017_01,Winter,0.500000,...,57738.246094,152.418533,5.504085,1.932987,89.4145,22.8093,True,15,0,True
2,CL509,Kapilmuni,22.6887,89.3088,2017,1,2017-01-01,2017_01,Winter,0.500000,...,48315.433594,157.559280,4.366783,1.917250,89.3088,22.6887,True,13,2,False
3,CL510,Khulna,22.8319,89.5500,2017,1,2017-01-01,2017_01,Winter,0.500000,...,56018.875000,154.137131,6.754479,2.105171,89.5500,22.8319,True,15,0,True
4,CL515,Paikgacha,22.5850,89.3182,2017,1,2017-01-01,2017_01,Winter,0.500000,...,37890.070312,129.930923,4.656047,1.600269,89.3182,22.5850,True,15,0,True
5,CL517,Rupsa,22.7900,89.5900,2017,1,2017-01-01,2017_01,Winter,0.500000,...,50599.093750,140.324051,3.710326,1.544958,89.5900,22.7900,True,15,0,True
6,CL503,Chalna,22.6012,89.5195,2017,2,2017-02-01,2017_02,Winter,0.866025,...,31467.640625,108.518059,3.491428,1.720395,89.5195,22.6012,True,13,2,False
7,CL504,Dumuria,22.8093,89.4145,2017,2,2017-02-01,2017_02,Winter,0.866025,...,57738.246094,152.418533,5.504085,1.932987,89.4145,22.8093,True,15,0,True
8,CL509,Kapilmuni,22.6887,89.3088,2017,2,2017-02-01,2017_02,Winter,0.866025,...,48315.433594,157.559280,4.366783,1.917250,89.3088,22.6887,True,13,2,False
9,CL510,Khulna,22.8319,89.5500,2017,2,2017-02-01,2017_02,Winter,0.866025,...,56018.875000,154.137131,6.754479,2.105171,89.5500,22.8319,True,15,0,True



MISSING VALUE SUMMARY


,Predictor,Valid_Values,Missing_Values,Missing_Percentage
0,CCS,360,72,16.6667
1,CDR,360,72,16.6667
2,CHIRPS,432,0,0.0000
3,ERA5,432,0,0.0000
4,GSMaP_Gauge,432,0,0.0000
5,GSMaP_MVK,414,18,4.1667
6,IMERG,432,0,0.0000
7,LST,385,47,10.8796
8,NDVI,432,0,0.0000
9,PDIR,360,72,16.6667



STATION SUMMARY


,station_id,station_name,records,valid_rainfall,mean_rainfall_mm,min_rainfall_mm,max_rainfall_mm,complete_predictor_rows,mean_valid_predictors,predictor_completeness_percent
0,CL503,Chalna,72,72,209.788889,0.0,783.0,0,12.763889,0.000000
1,CL504,Dumuria,72,72,131.883333,0.0,704.3,60,14.833333,83.333333
2,CL509,Kapilmuni,72,72,149.973611,0.0,606.0,0,12.888889,0.000000
3,CL510,Khulna,72,72,159.333333,0.0,720.0,60,14.833333,83.333333
4,CL515,Paikgacha,72,72,161.234722,0.0,687.0,65,14.902778,90.277778
5,CL517,Rupsa,72,72,140.090278,0.0,587.5,63,14.875000,87.500000



GAUGE VALUE EXTRACTION COMPLETED
Successful months : 72
Failed months     : 0
Final rows        : 432
Clean ML rows     : 432
Predictor columns : 15
GeoPackage saved  : True

Output files:
1. E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\model_data\gauge_predictor_dataset_2017_2022.csv
2. E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\model_data\gauge_predictor_dataset_clean.csv
3. E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\model_data\gauge_predictor_dataset_2017_2022.gpkg

Log files:
1. E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\logs\gauge_value_extraction_log.csv
2. E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\logs\gauge_value_extraction_summary.csv
3. E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\logs\gauge_value_missing_summary